In [0]:
from pyspark.sql import functions as F

df_bronze_articles = spark.table("workspace.default.bronze_nyt_articles")

# Limpeza: remove registros sem conteúdo essencial (o achado de qualidade de dados) e duplicatas
df_silver_articles = (df_bronze_articles
    .filter(F.col("web_url").isNotNull() & F.col("headline").isNotNull())
    .dropDuplicates(["web_url"])
    .withColumn("published_date", F.to_timestamp("published_date"))
    .withColumn("published_day", F.to_date("published_date"))
)

print(f"Bronze: {df_bronze_articles.count()} | Silver: {df_silver_articles.count()}")
df_silver_articles = (df_bronze_articles
    .filter(F.col("web_url").isNotNull() & F.col("headline").isNotNull())
    .dropDuplicates(["web_url"])
    .withColumn("published_date", F.try_to_timestamp("published_date"))
    .withColumn("published_day", F.to_date("published_date"))
)

n_bronze = df_bronze_articles.count()
n_data_invalida = df_silver_articles.filter(F.col("published_date").isNull()).count()
print(f"Bronze: {n_bronze}")
print(f"Registros com published_date inválido (desalinhamento no CSV): {n_data_invalida}")

df_silver_articles = df_silver_articles.filter(F.col("published_date").isNotNull())
print(f"Silver final: {df_silver_articles.count()}")

df_silver_articles.write.mode("overwrite").saveAsTable("workspace.default.silver_nyt_articles")

In [0]:
%pip install vaderSentiment

In [0]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()
pdf = spark.table("workspace.default.silver_nyt_articles").toPandas()

def calc_sentiment(row):
    texto = " ".join([str(t) for t in [row.get("headline"), row.get("abstract"), row.get("snippet")] if t])
    return analyzer.polarity_scores(texto)["compound"]

pdf["sentiment"] = pdf.apply(calc_sentiment, axis=1)

df_silver_articles = spark.createDataFrame(pdf)
df_silver_articles.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.silver_nyt_articles")
print("Sentimento calculado. Amostra:")
df_silver_articles.select("headline", "sentiment").show(5, truncate=60)

In [0]:
import re

def limpar_colunas_ticker(df):
    novo_df = df
    for col_name in df.columns:
        novo_nome = re.sub(r"_.*$", "", col_name) if col_name != "Date" else col_name
        if novo_nome != col_name:
            novo_df = novo_df.withColumnRenamed(col_name, novo_nome)
    return novo_df

df_silver_dxy = (limpar_colunas_ticker(spark.table("workspace.default.bronze_dxy"))
    .withColumn("Date", F.to_date("Date"))
    .orderBy("Date"))
df_silver_usdbrl = (limpar_colunas_ticker(spark.table("workspace.default.bronze_usdbrl"))
    .withColumn("Date", F.to_date("Date"))
    .orderBy("Date"))

df_silver_dxy.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.silver_dxy")
df_silver_usdbrl.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.silver_usdbrl")

df_silver_dxy.printSchema()

In [0]:
from pyspark.sql.window import Window

df_news_daily = (spark.table("workspace.default.silver_nyt_articles")
    .groupBy("published_day")
    .agg(F.count("*").alias("total_artigos"), F.avg("sentiment").alias("sentimento_medio"))
    .withColumnRenamed("published_day", "dia")
)

w = Window.orderBy("Date")
df_dxy_ret = (spark.table("workspace.default.silver_dxy")
    .withColumn("dxy_return", (F.col("Close") - F.lag("Close", 1).over(w)) / F.lag("Close", 1).over(w))
    .select(F.col("Date").alias("dia"), F.col("Close").alias("dxy_close"), "dxy_return")
)
df_usdbrl_ret = (spark.table("workspace.default.silver_usdbrl")
    .withColumn("usdbrl_return", (F.col("Close") - F.lag("Close", 1).over(w)) / F.lag("Close", 1).over(w))
    .select(F.col("Date").alias("dia"), F.col("Close").alias("usdbrl_close"), "usdbrl_return")
)

df_gold = (df_news_daily
    .join(df_dxy_ret, on="dia", how="inner")
    .join(df_usdbrl_ret, on="dia", how="inner")
    .withColumn("sentimento_lag1", F.lag("sentimento_medio", 1).over(Window.orderBy("dia")))
    .orderBy("dia")
)

df_gold.write.mode("overwrite").saveAsTable("workspace.default.gold_analise_diaria")
df_gold.display()

In [0]:
print(df_gold.count())

In [0]:
import pandas as pd
import matplotlib.pyplot as plt

df_gold = spark.table("workspace.default.gold_analise_diaria").toPandas()
df_gold = df_gold.sort_values("dia").reset_index(drop=True)
df_gold["dxy_vol"] = df_gold["dxy_return"].abs()

print(df_gold.describe())

# P1: volume de notícias x volatilidade do DXY (proxy: |retorno diário|)
corr_p1 = df_gold["total_artigos"].corr(df_gold["dxy_vol"])
# P2: sentimento médio x retorno do DXY no mesmo dia
corr_p2 = df_gold["sentimento_medio"].corr(df_gold["dxy_return"])
# P3: sentimento do dia anterior (lag) x retorno do DXY
corr_p3 = df_gold["sentimento_lag1"].corr(df_gold["dxy_return"])
# P4: mesmo padrão (P2/P3) aplicado ao USDBRL
corr_p4_same_day = df_gold["sentimento_medio"].corr(df_gold["usdbrl_return"])
corr_p4_lag = df_gold["sentimento_lag1"].corr(df_gold["usdbrl_return"])

print(f"P1 - volume de artigos x volatilidade DXY: {corr_p1:.3f}")
print(f"P2 - sentimento médio x retorno DXY (mesmo dia): {corr_p2:.3f}")
print(f"P3 - sentimento (dia anterior) x retorno DXY: {corr_p3:.3f}")
print(f"P4 - sentimento médio x retorno USDBRL (mesmo dia): {corr_p4_same_day:.3f}")
print(f"P4 - sentimento (dia anterior) x retorno USDBRL: {corr_p4_lag:.3f}")

In [0]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].scatter(df_gold["total_artigos"], df_gold["dxy_vol"], alpha=0.5)
axes[0,0].set_title(f"P1: Volume de notícias x Volatilidade DXY (corr={corr_p1:.2f})")
axes[0,0].set_xlabel("Total de artigos/dia"); axes[0,0].set_ylabel("|Retorno DXY|")

axes[0,1].scatter(df_gold["sentimento_medio"], df_gold["dxy_return"], alpha=0.5, color="orange")
axes[0,1].set_title(f"P2: Sentimento médio x Retorno DXY (corr={corr_p2:.2f})")
axes[0,1].set_xlabel("Sentimento médio (VADER)"); axes[0,1].set_ylabel("Retorno DXY")

axes[1,0].scatter(df_gold["sentimento_lag1"], df_gold["dxy_return"], alpha=0.5, color="green")
axes[1,0].set_title(f"P3: Sentimento (dia anterior) x Retorno DXY (corr={corr_p3:.2f})")
axes[1,0].set_xlabel("Sentimento médio (lag 1 dia)"); axes[1,0].set_ylabel("Retorno DXY")

axes[1,1].scatter(df_gold["sentimento_medio"], df_gold["usdbrl_return"], alpha=0.5, color="red")
axes[1,1].set_title(f"P4: Sentimento médio x Retorno USDBRL (corr={corr_p4_same_day:.2f})")
axes[1,1].set_xlabel("Sentimento médio (VADER)"); axes[1,1].set_ylabel("Retorno USDBRL")

plt.tight_layout()
plt.show()

In [0]:
from scipy.stats import pearsonr

def corr_test(colx, coly, label):
    sub = df_gold[[colx, coly]].dropna()
    r, p = pearsonr(sub[colx], sub[coly])
    sig = "significativo (p<0.05)" if p < 0.05 else "NÃO significativo (p>=0.05)"
    print(f"{label} (n={len(sub)}): r={r:.3f}, p={p:.3f} -> {sig}")

corr_test("total_artigos", "dxy_vol", "P1")
corr_test("sentimento_medio", "dxy_return", "P2")
corr_test("sentimento_lag1", "dxy_return", "P3")
corr_test("sentimento_medio", "usdbrl_return", "P4 (mesmo dia)")
corr_test("sentimento_lag1", "usdbrl_return", "P4 (lag)")